# Healthcare Claims Fraud & Cost Analytics Platform

### Problem Statement
 The goal of this project is to analyze healthcare claims to detect abnormal providers behaviour and healthcare cost patterns.

Think like this:

A health insurance company wants to know:

- Which doctors are suspicious?
- Which treatments cost the most?
- Which patients generate the highest costs?

Because fraud costs billions in healthcare.

The job is to investigate the data.

Load the dataset

In [2]:
import pandas as pd
beneficiary = pd.read_csv(r"D:\Data Analysis\Projects\HEALTHCARE PROVIDER FRAUD DETECTION ANALYSIS\Data\Train_Beneficiarydata-1542865627584.csv")
inpatient = pd.read_csv(r"D:\Data Analysis\Projects\HEALTHCARE PROVIDER FRAUD DETECTION ANALYSIS\Data\Train_Inpatientdata-1542865627584.csv")
outpatient = pd.read_csv(r"D:\Data Analysis\Projects\HEALTHCARE PROVIDER FRAUD DETECTION ANALYSIS\Data\Train_Outpatientdata-1542865627584.csv")
provider = pd.read_csv(r"D:\Data Analysis\Projects\HEALTHCARE PROVIDER FRAUD DETECTION ANALYSIS\Data\Train-1542865627584.csv")

- Beneficiary are the patients
- inpatient are the patients who stay overnight in the hospital
- outpatient are the patients who visits the hospital but doesn't stay overnight
- providers are the doctors/hospital

Check Dataset Size

In [3]:
print("beneficiary:",beneficiary.shape)
print("inpatient:",inpatient.shape)
print("outpatient:",outpatient.shape)
print("provider:",provider.shape)

beneficiary: (138556, 25)
inpatient: (40474, 30)
outpatient: (517737, 27)
provider: (5410, 2)


- 40474 + 517737 = 558211 healthcare claims i.e insurance claims
- 138556 patients i.e people receiving traement
- 5410 providers i.e doctors/hospital
- 558k claims / 138k patients ≈ 4 claims per patient

Check Columns

In [4]:
print(f"beneficiery: {beneficiary.columns}")

beneficiery: Index(['BeneID', 'DOB', 'DOD', 'Gender', 'Race', 'RenalDiseaseIndicator',
       'State', 'County', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov',
       'ChronicCond_Alzheimer', 'ChronicCond_Heartfailure',
       'ChronicCond_KidneyDisease', 'ChronicCond_Cancer',
       'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression',
       'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart',
       'ChronicCond_Osteoporasis', 'ChronicCond_rheumatoidarthritis',
       'ChronicCond_stroke', 'IPAnnualReimbursementAmt',
       'IPAnnualDeductibleAmt', 'OPAnnualReimbursementAmt',
       'OPAnnualDeductibleAmt'],
      dtype='object')


In [5]:
print(f"inpatient: {inpatient.columns}")

inpatient: Index(['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider',
       'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician',
       'OtherPhysician', 'AdmissionDt', 'ClmAdmitDiagnosisCode',
       'DeductibleAmtPaid', 'DischargeDt', 'DiagnosisGroupCode',
       'ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10', 'ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6'],
      dtype='object')


These are the columns that doesn't exist in the outpatient table:
- 'AdmissionDt' = admission date
- 'DischargeDt' = discharge date
- 'DiagnosisGroupCode' = category of hospital treatment when admmitted, A Diagnosis Related Group is a way hospitals classify treatments into standard billing categories. Instead of billing every service separately, hospitals group treatments into medical categories.


In [6]:
print(f"outpatient: {outpatient.columns}")

outpatient: Index(['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider',
       'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician',
       'OtherPhysician', 'ClmDiagnosisCode_1', 'ClmDiagnosisCode_2',
       'ClmDiagnosisCode_3', 'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5',
       'ClmDiagnosisCode_6', 'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8',
       'ClmDiagnosisCode_9', 'ClmDiagnosisCode_10', 'ClmProcedureCode_1',
       'ClmProcedureCode_2', 'ClmProcedureCode_3', 'ClmProcedureCode_4',
       'ClmProcedureCode_5', 'ClmProcedureCode_6', 'DeductibleAmtPaid',
       'ClmAdmitDiagnosisCode'],
      dtype='object')


In [7]:
print(f"provider: {provider.columns}")

provider: Index(['Provider', 'PotentialFraud'], dtype='object')


Inspect Sample Data

In [8]:
beneficiary.head()

,BeneID,DOB,DOD,Gender,Race,RenalDiseaseIndicator,State,County,NoOfMonths_PartACov,NoOfMonths_PartBCov,...,ChronicCond_Depression,ChronicCond_Diabetes,ChronicCond_IschemicHeart,ChronicCond_Osteoporasis,ChronicCond_rheumatoidarthritis,ChronicCond_stroke,IPAnnualReimbursementAmt,IPAnnualDeductibleAmt,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt
0,BENE11001,1943-01-01,NaN,1,1,0,39,230,12,12,...,1,1,1,2,1,1,36000,3204,60,70
1,BENE11002,1936-09-01,NaN,2,1,0,39,280,12,12,...,2,2,2,2,2,2,0,0,30,50
2,BENE11003,1936-08-01,NaN,1,1,0,52,590,12,12,...,2,2,1,2,2,2,0,0,90,40
3,BENE11004,1922-07-01,NaN,1,1,0,39,270,12,12,...,2,1,1,1,1,2,0,0,1810,760
4,BENE11005,1935-09-01,NaN,1,1,0,24,680,12,12,...,2,1,2,2,2,2,0,0,1790,1200


- Gender = (1: Male, 2: Female)
- RenalDiseaseIndicator = (Y: yes, 0: no)
- NoOfMonths_PartACov = Months patient had medicalcare. Part A covers hospital stays, inpatient care
- NoOfMonths_PartBCov = Months patient had medicalcare. Part B covers doctor visits, outpatient services
- Chronic Conditions = These columns indicate whether patient has long-term illnesses. (1: yes, 2: no)
- IPAnnualReimbursementAmt = Total inpatient reimbursement amount paid for the patient.
- IPAnnualDeductibleAmt = Total inpatient deductible (fixed amount a policy holder must pay out of pocket) paid by patient.
- OPAnnualReimbursementAmt = Total outpatient reimbursement amount.
- OPAnnualDeductibleAmt = Total outpatient deductible paid.

In [9]:
inpatient.head()

,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,OtherPhysician,AdmissionDt,...,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6
0,BENE11001,CLM46614,2009-04-12,2009-04-18,PRV55912,26000,PHY390922,NaN,NaN,2009-04-12,...,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BENE11001,CLM66048,2009-08-31,2009-09-02,PRV55907,5000,PHY318495,PHY318495,NaN,2009-08-31,...,NaN,NaN,NaN,NaN,7092.0,NaN,NaN,NaN,NaN,NaN
2,BENE11001,CLM68358,2009-09-17,2009-09-20,PRV56046,5000,PHY372395,NaN,PHY324689,2009-09-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BENE11011,CLM38412,2009-02-14,2009-02-22,PRV52405,5000,PHY369659,PHY392961,PHY349768,2009-02-14,...,25062,40390,4019,NaN,331.0,NaN,NaN,NaN,NaN,NaN
4,BENE11014,CLM63689,2009-08-13,2009-08-30,PRV56614,10000,PHY379376,PHY398258,NaN,2009-08-13,...,5119,29620,20300,NaN,3893.0,NaN,NaN,NaN,NaN,NaN


- ClmAdmitDiagnosisCode = Primary diagnosis at admission, but stored in code numbers.
- DeductibleAmtPaid = Amount patient paid as deductible.
- DiagnosisGroupCode = DRG code (treatment category). Example: joint replacement, heart failure treatment
- Diagnosis Codes = These represent medical conditions diagnosed.
- Procedure Codes = These represent medical procedures performed.

In [10]:
outpatient.head()

,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,OtherPhysician,ClmDiagnosisCode_1,...,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,DeductibleAmtPaid,ClmAdmitDiagnosisCode
0,BENE11002,CLM624349,2009-10-11,2009-10-11,PRV56011,30,PHY326117,NaN,NaN,78943,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,56409
1,BENE11003,CLM189947,2009-02-12,2009-02-12,PRV57610,80,PHY362868,NaN,NaN,6115,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,79380
2,BENE11003,CLM438021,2009-06-27,2009-06-27,PRV57595,10,PHY328821,NaN,NaN,2723,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
3,BENE11004,CLM121801,2009-01-06,2009-01-06,PRV56011,40,PHY334319,NaN,NaN,71988,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
4,BENE11004,CLM150998,2009-01-22,2009-01-22,PRV56011,200,PHY403831,NaN,NaN,82382,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,71947


In [11]:
provider.head()

,Provider,PotentialFraud
0,PRV51001,No
1,PRV51003,Yes
2,PRV51004,No
3,PRV51005,Yes
4,PRV51007,No


PotentialFraud = Indicates whether provider is flagged for fraud.

Values:
- Yes
- No

This column allows you to compare:
- fraudulent providers
- legitimate providers

Merging inpatient and outpatient into claims

In [12]:
claims = pd.concat([inpatient,outpatient], axis = 0)
claims.shape

(558211, 30)

In [13]:
claims.describe()

,InscClaimAmtReimbursed,DeductibleAmtPaid,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6
count,558211.000000,557312.000000,23310.000000,5490.000000,969.000000,118.000000,9.000000,0.0
mean,997.012133,78.421085,5896.154612,4106.358106,4221.123839,4070.262712,5269.444444,NaN
std,3821.534891,274.016812,3050.489933,2031.640878,2281.849885,2037.626990,2780.071632,NaN
min,0.000000,0.000000,11.000000,42.000000,42.000000,42.000000,2724.000000,NaN
25%,40.000000,0.000000,3848.000000,2724.000000,2724.000000,2754.250000,4139.000000,NaN
50%,80.000000,0.000000,5363.000000,4019.000000,4019.000000,4019.000000,4139.000000,NaN
75%,300.000000,0.000000,8669.000000,4439.000000,5185.000000,4439.000000,5185.000000,NaN
max,125000.000000,1068.000000,9999.000000,9999.000000,9999.000000,9986.000000,9982.000000,NaN


Merging claims with provider

In [14]:
claims_provider = claims.merge(provider, on = "Provider", how="left")
claims_provider.head()

,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,OtherPhysician,AdmissionDt,...,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,PotentialFraud
0,BENE11001,CLM46614,2009-04-12,2009-04-18,PRV55912,26000,PHY390922,NaN,NaN,2009-04-12,...,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes
1,BENE11001,CLM66048,2009-08-31,2009-09-02,PRV55907,5000,PHY318495,PHY318495,NaN,2009-08-31,...,NaN,NaN,NaN,7092.0,NaN,NaN,NaN,NaN,NaN,No
2,BENE11001,CLM68358,2009-09-17,2009-09-20,PRV56046,5000,PHY372395,NaN,PHY324689,2009-09-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No
3,BENE11011,CLM38412,2009-02-14,2009-02-22,PRV52405,5000,PHY369659,PHY392961,PHY349768,2009-02-14,...,40390,4019,NaN,331.0,NaN,NaN,NaN,NaN,NaN,No
4,BENE11014,CLM63689,2009-08-13,2009-08-30,PRV56614,10000,PHY379376,PHY398258,NaN,2009-08-13,...,29620,20300,NaN,3893.0,NaN,NaN,NaN,NaN,NaN,No


In [15]:
claims_provider.shape

(558211, 31)

claims_provider


├─ Claim information

├─ Patient ID

├─ Provider ID

└─ Fraud label

Now we can tell whether this claim submitted by a fraudulent provider or not?

Compare Claim Amount by PotentialFraud

In [16]:
claims_provider.groupby("PotentialFraud")["InscClaimAmtReimbursed"].describe()

,count,mean,std,min,25%,50%,75%,max
PotentialFraud,,,,,,,,
No,345415.0,755.213352,3056.460166,0.0,40.0,80.0,300.0,125000.0
Yes,212796.0,1389.505066,4785.074685,0.0,40.0,90.0,400.0,125000.0


- Non-fraud providers submitted 345k claims
- Fraud providers submitted 212k claims

This already tells us fraud providers are quite active.

- Fraudulent providers submit claims ~84% higher than legitimate providers on average. Calculation -> ((1389-755)/755)*100 = 83.9
- Some fraud providers submit very large claims occasionally, as the fraud providers show much higher variability in claim amounts.

Compare claim by providers

In [17]:
claims_provider.groupby("PotentialFraud")["Provider"].nunique()
# nunique() means: number of unique values

PotentialFraud
No     4904
Yes     506
Name: Provider, dtype: int64

(506/(4904 + 506))*100 = 9.35%
- Roughly 9% of providers are practicing potentaial fraud

Calculate Claim Per Provider

In [18]:
claims_per_provider = claims_provider.groupby("PotentialFraud").size()
claims_per_provider

''' .size()
Counts how many rows exist in each group.
Since each row = one claim
'''

' .size()\nCounts how many rows exist in each group.\nSince each row = one claim\n'

(212796/(345415 + 212796))*100 = 38.12%
- Roughly 38% of claims are potential fraud done by 9% of the providers

Average Claims per provider per claim

In [19]:
providers_count = claims_provider.groupby("PotentialFraud")["Provider"].nunique()
claims_count = claims_provider.groupby("PotentialFraud").size()
claims_per_provider = claims_count / providers_count #claims per provider = total claims / number of providers
'''
345415/4906 = 70.43
212796/504 = 420.54
'''
claims_per_provider

PotentialFraud
No      70.435359
Yes    420.545455
dtype: float64

For non-fradulant provders:
- 345415/4906 ~ 70 claim per provider

For fradulant providers:
- 212796/504 ~ 420 claim per provider

Claims per diagnosis

In [20]:
diag_cols = ['ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10']

#creating list of diagnosis because to group by we need to use them together

In [21]:
diagnosis_long = claims_provider.melt(
    id_vars=['ClaimID','Provider','PotentialFraud','InscClaimAmtReimbursed'],
    value_vars=diag_cols,
    var_name='Diagnosis_Position',
    value_name='DiagnosisCode'
)

'''
claims_provider.melt(...)

melt() reshapes data.

It converts:

multiple columns → single column
id_vars=['ClaimID','Provider','PotentialFraud','InscClaimAmtReimbursed']

These columns are kept as-is.

Meaning:

Each diagnosis will still be linked to:

claim
provider
fraud status
insurance claim amount

value_vars=diag_cols

These are the columns we are transforming:

ClmDiagnosisCode_1 → ClmDiagnosisCode_10

var_name='Diagnosis_Position'

This tells us:

Was it diagnosis 1? diagnosis 2? etc.

value_name='DiagnosisCode'

This is the actual diagnosis value.
'''

"\nclaims_provider.melt(...)\n\nmelt() reshapes data.\n\nIt converts:\n\nmultiple columns → single column\nid_vars=['ClaimID','Provider','PotentialFraud','InscClaimAmtReimbursed']\n\nThese columns are kept as-is.\n\nMeaning:\n\nEach diagnosis will still be linked to:\n\nclaim\nprovider\nfraud status\ninsurance claim amount\n\nvalue_vars=diag_cols\n\nThese are the columns we are transforming:\n\nClmDiagnosisCode_1 → ClmDiagnosisCode_10\n\nvar_name='Diagnosis_Position'\n\nThis tells us:\n\nWas it diagnosis 1? diagnosis 2? etc.\n\nvalue_name='DiagnosisCode'\n\nThis is the actual diagnosis value.\n"

In [22]:
diagnosis_long = diagnosis_long.dropna(subset=['DiagnosisCode'])
#many diagnosis cloumns are empty, we remove them where DiagnosisCode = NaN

In [23]:
diag_counts = diagnosis_long.groupby(["PotentialFraud","DiagnosisCode"]).size().reset_index(name = 'ClaimCount')
'''
groupby(['PotentialFraud','DiagnosisCode'])

We group by:

fraud status
diagnosis code
.size()

Counts number of rows → number of claims

.reset_index(name='ClaimCount')

Converts result into a proper table with column:

ClaimCount
'''

"\ngroupby(['PotentialFraud','DiagnosisCode'])\n\nWe group by:\n\nfraud status\ndiagnosis code\n.size()\n\nCounts number of rows → number of claims\n\n.reset_index(name='ClaimCount')\n\nConverts result into a proper table with column:\n\nClaimCount\n"

In [24]:
diag_counts.sort_values(
    ['PotentialFraud','ClaimCount'],
    ascending=[True, False]
).groupby('PotentialFraud').head(10)

'''
diag_counts

This table contains:

PotentialFraud | DiagnosisCode | ClaimCount

Meaning:

how many claims exist for each diagnosis, split by fraud vs non-fraud.

.sort_values(...)
['PotentialFraud','ClaimCount']

We sort by:

Fraud group (No first, then Yes)
Claim count (highest first)
ascending=[True, False]

Means:

True → PotentialFraud sorted normally (No → Yes)
False → ClaimCount sorted descending (biggest first)

.groupby('PotentialFraud')

Now we separate:

Group 1 → No
Group 2 → Yes

.head(10)

Take top 10 rows from each group

So final result =

Top 10 diagnoses for:
- Non-fraud providers
- Fraud providers
'''

"\ndiag_counts\n\nThis table contains:\n\nPotentialFraud | DiagnosisCode | ClaimCount\n\nMeaning:\n\nhow many claims exist for each diagnosis, split by fraud vs non-fraud.\n\n.sort_values(...)\n['PotentialFraud','ClaimCount']\n\nWe sort by:\n\nFraud group (No first, then Yes)\nClaim count (highest first)\nascending=[True, False]\n\nMeans:\n\nTrue → PotentialFraud sorted normally (No → Yes)\nFalse → ClaimCount sorted descending (biggest first)\n\n.groupby('PotentialFraud')\n\nNow we separate:\n\nGroup 1 → No\nGroup 2 → Yes\n\n.head(10)\n\nTake top 10 rows from each group\n\nSo final result =\n\nTop 10 diagnoses for:\n- Non-fraud providers\n- Fraud providers\n"

In [25]:
diag_counts

,PotentialFraud,DiagnosisCode,ClaimCount
0,No,0010,5
1,No,0019,2
2,No,0020,3
3,No,0021,1
4,No,0022,1
...,...,...,...
20181,Yes,V8902,8
20182,Yes,V8903,11
20183,Yes,V8904,14
20184,Yes,V8905,12


In [26]:
diag_amount = diagnosis_long.groupby(
    ['PotentialFraud','DiagnosisCode']
)['InscClaimAmtReimbursed'].mean().reset_index(name = 'AvgClaimAmount')
diag_amount

,PotentialFraud,DiagnosisCode,AvgClaimAmount
0,No,0010,884.000000
1,No,0019,5.000000
2,No,0020,210.000000
3,No,0021,600.000000
4,No,0022,70.000000
...,...,...,...
20181,Yes,V8902,221.250000
20182,Yes,V8903,246.363636
20183,Yes,V8904,117.142857
20184,Yes,V8905,220.833333


In [27]:
diag_amount.sort_values(
    ['PotentialFraud','AvgClaimAmount'],
    ascending=[True, False]
).groupby('PotentialFraud').head(10)

,PotentialFraud,DiagnosisCode,AvgClaimAmount
9302,No,E9190,36000.000000
8092,No,86502,31000.000000
8309,No,90241,31000.000000
9221,No,E8708,29200.000000
8314,No,90301,29000.000000
9369,No,E9344,29000.000000
88,No,01500,28550.000000
4422,No,5185,26896.518987
8070,No,8604,26333.333333
4690,No,53250,26006.666667


- Fraud diagnoses → ₹57,000 avg
- Non-fraud diagnoses → ₹26k–36k avg
The above data shows the average claims without count, so trusting the data without verification will be misleading.

So, we need both number of claims and average claim amount

In [28]:
diag_summary = diagnosis_long.groupby(
    ['PotentialFraud','DiagnosisCode']
).agg(
    ClaimCount=('DiagnosisCode','count'),
    AvgClaimAmount=('InscClaimAmtReimbursed','mean')
).reset_index()

'''
groupby(['PotentialFraud','DiagnosisCode'])

Group by:

fraud status
diagnosis
.agg(...)

Aggregate multiple metrics:

ClaimCount=('DiagnosisCode','count')

Count how many times diagnosis appears.

AvgClaimAmount=('InscClaimAmtReimbursed','mean')

Average claim value.
'''

"\ngroupby(['PotentialFraud','DiagnosisCode'])\n\nGroup by:\n\nfraud status\ndiagnosis\n.agg(...)\n\nAggregate multiple metrics:\n\nClaimCount=('DiagnosisCode','count')\n\nCount how many times diagnosis appears.\n\nAvgClaimAmount=('InscClaimAmtReimbursed','mean')\n\nAverage claim value.\n"

In [29]:
diag_summary = diag_summary[diag_summary['ClaimCount'] > 100]
'''
we only keep claims more than 100 so that the results are reliable
'''

'\nwe only keep claims more than 100 so that the results are reliable\n'

In [30]:
diag_summary.sort_values(
    ['PotentialFraud','AvgClaimAmount'],
    ascending=[True, False]
).groupby('PotentialFraud').head(10)

,PotentialFraud,DiagnosisCode,ClaimCount,AvgClaimAmount
4422,No,5185,158,26896.518987
9087,No,9971,136,20295.735294
9014,No,99592,287,17893.135889
7049,No,78552,226,17299.823009
4425,No,51881,879,15694.755404
9103,No,99811,114,14563.245614
9091,No,9974,119,14467.563025
3860,No,41071,500,13349.600000
182,No,0389,806,13108.945409
1749,No,261,129,12990.232558


In [31]:
diag_summary

,PotentialFraud,DiagnosisCode,ClaimCount,AvgClaimAmount
58,No,00845,339,10378.849558
145,No,03284,195,140.307692
182,No,0389,806,13108.945409
202,No,04104,123,7448.617886
206,No,04111,197,3749.238579
...,...,...,...,...
20130,Yes,V829,125,96.320000
20142,Yes,V850,218,9903.256881
20159,Yes,V854,486,10030.370370
20164,Yes,V860,116,946.896552


In [32]:
diag_summary['DiagnosisCode'].nunique()

1381

In [33]:
fraud_diag = diag_summary[diag_summary['PotentialFraud'] == 'Yes']
nonfraud_diag = diag_summary[diag_summary['PotentialFraud'] == 'No']
#separating fraud vs non fraud to compare them

In [34]:
fraud_top = fraud_diag.sort_values(
    ['ClaimCount','AvgClaimAmount'],
    ascending=[False,False]
)
fraud_top.head(10)

,PotentialFraud,DiagnosisCode,ClaimCount,AvgClaimAmount
13985,Yes,4019,31029,2622.305263
11921,Yes,25000,15087,2892.788493
12090,Yes,2724,14690,2812.656229
19845,Yes,V5869,9116,532.757789
14147,Yes,42731,8698,4102.954702
13984,Yes,4011,8652,280.562876
19838,Yes,V5861,7629,891.599161
11887,Yes,2449,7276,3146.990104
12086,Yes,2720,7121,2034.450218
14158,Yes,4280,7115,5234.229093


In [35]:
nonfraud_top = nonfraud_diag.sort_values(
    ['ClaimCount','AvgClaimAmount'],
    ascending=[False,False]
)
nonfraud_top.head(10)

,PotentialFraud,DiagnosisCode,ClaimCount,AvgClaimAmount
3800,No,4019,46027,1407.302236
1636,No,25000,22269,1512.711842
1805,No,2724,21073,1510.446543
10077,No,V5869,15788,361.698759
3799,No,4011,15121,191.809404
10070,No,V5861,12372,456.934206
3962,No,42731,11440,2279.986014
1801,No,2720,11147,1124.351844
1602,No,2449,10324,1669.019760
4654,No,53081,8467,2461.796386


- Top fraud is not happening by introducing new diagnosis, it is happening within common diagnosis. i.e. the top diagnosises are appearing in both the fraud and non-fraud claims.
- Even the total fraud claims for top diagnosis are less than non-fraud claims, the fraud providers are charging between 45% to 91% more than non-fraudulant providers.
- Fraudulent providers do not rely on rare or unique diagnoses but instead inflate reimbursement amounts within commonly occurring diagnosis categories.

Do fraud providers perform more procedures per claim

In [42]:
#select all procedure columns to one single column

proc_cols = [
    'ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6'
]

In [ ]:
#count procedures per claim (not null values)

claims_provider['ProcedureCount'] = claims_provider[proc_cols].notna().sum(axis=1)

'''
claims_provider[proc_cols]

 Select only procedure columns

So table becomes like:

Proc1	Proc2	Proc3
A	    NaN     B

.notna()

Checks if value is NOT empty

Example
Proc1	Proc2	Proc3
A	    NaN 	B

Becomes:

| True | False | True |

.sum(axis=1)

Add values row-wise

Remember:

True = 1
False = 0

So:

1 + 0 + 1 = 2

Final result:
New column created ProcedureCount
'''

In [46]:
claims_provider.groupby('PotentialFraud')['ProcedureCount'].mean()

PotentialFraud
No     0.036133
Yes    0.081839
Name: ProcedureCount, dtype: float64

- Fraudulent providers perform significantly more procedures per claim (~2.3x higher), indicating potential overutilization of medical procedures to inflate billing. 0.081/0.036 ~ 2.25


Identify Suspecious Providers

In [48]:
provider_summary = claims_provider.groupby(['Provider','PotentialFraud']).agg(
    TotalClaims=('ClaimID','count'),
    AvgClaimAmount=('InscClaimAmtReimbursed','mean'),
    AvgProcedures=('ProcedureCount','mean')
).reset_index()

provider_summary.head()

,Provider,PotentialFraud,TotalClaims,AvgClaimAmount,AvgProcedures
0,PRV51001,No,25,4185.600000,0.120000
1,PRV51003,Yes,132,4588.409091,0.363636
2,PRV51004,No,149,350.134228,0.000000
3,PRV51005,Yes,1165,241.124464,0.000000
4,PRV51007,No,72,468.194444,0.013889


In [68]:
provider_summary[provider_summary['PotentialFraud']=='Yes'].sort_values(
    by=['AvgClaimAmount','AvgProcedures'],
    ascending=[False,False]
).head(15)

,Provider,PotentialFraud,TotalClaims,AvgClaimAmount,AvgProcedures
645,PRV51805,Yes,3,22333.333333,0.666667
2646,PRV54297,Yes,8,21000.000000,1.375000
1156,PRV52445,Yes,5,20400.000000,1.000000
4726,PRV56918,Yes,7,20285.714286,1.000000
3096,PRV54876,Yes,3,19333.333333,1.333333
3306,PRV55147,Yes,6,18700.000000,0.666667
1463,PRV52815,Yes,26,15000.000000,0.615385
4854,PRV57096,Yes,11,14545.454545,1.000000
2953,PRV54680,Yes,18,14277.777778,1.222222
1905,PRV53379,Yes,11,13909.090909,1.090909


In [72]:
provider_summary[provider_summary['PotentialFraud']=='Yes'].sort_values(
    by=['TotalClaims'],
    ascending=False
).head(25)

,Provider,PotentialFraud,TotalClaims,AvgClaimAmount,AvgProcedures
363,PRV51459,Yes,8240,281.782767,0.000728
2250,PRV53797,Yes,4739,275.079131,0.000422
455,PRV51574,Yes,4444,288.436094,0.000225
2335,PRV53918,Yes,3588,282.750836,0.001115
3113,PRV54895,Yes,3436,307.802678,0.002328
3363,PRV55215,Yes,3393,673.315650,0.034483
853,PRV52064,Yes,2844,393.699015,0.008439
4004,PRV56011,Yes,2833,261.786092,0.000353
3195,PRV55004,Yes,2399,293.293039,0.002918
5032,PRV57306,Yes,2315,271.904968,0.000864


In [62]:
provider_summary[provider_summary['PotentialFraud']=='Yes']['TotalClaims'].describe()

count     506.000000
mean      420.545455
std       722.734485
min         1.000000
25%        62.000000
50%       155.500000
75%       432.000000
max      8240.000000
Name: TotalClaims, dtype: float64

In [63]:
provider_summary[provider_summary['PotentialFraud']=='No']['TotalClaims'].describe()

count    4904.000000
mean       70.435359
std       128.942510
min         1.000000
25%         9.000000
50%        27.000000
75%        72.000000
max      1245.000000
Name: TotalClaims, dtype: float64

For fraud providers:

- Fraud providers are practicing ~420 claims on average per patient
- 8240 is the highest claims made by per patient
- 75% of the fraud providers have claims more than 432

For Non-fraud providers:

- Non-fraud providers are practicing ~70 caims on average per patient
- 1245 is the max claims done by the non-fraud providers

In [70]:
filtered_providers = provider_summary[
    (provider_summary['PotentialFraud']=='Yes')&
    (provider_summary['TotalClaims']>20)
]

filtered_providers.sort_values(
    by='AvgClaimAmount',
    ascending=False
).head(10)

,Provider,PotentialFraud,TotalClaims,AvgClaimAmount,AvgProcedures
1463,PRV52815,Yes,26,15000.000000,0.615385
2220,PRV53763,Yes,58,12741.379310,0.862069
662,PRV51826,Yes,21,12619.047619,0.809524
1693,PRV53114,Yes,29,12575.172414,0.517241
5181,PRV57481,Yes,62,12516.129032,0.935484
4102,PRV56134,Yes,62,12387.096774,0.774194
1035,PRV52292,Yes,25,12120.000000,0.880000
4820,PRV57049,Yes,22,12116.363636,0.727273
1973,PRV53461,Yes,36,11807.777778,0.305556
938,PRV52173,Yes,119,11608.403361,0.865546
